In [1]:
import pandas as pd

In [2]:
mapping_facilities = mapping = {
    "전문 소믈리에": "와인 페어링",
    "남/녀 화장실 구분" : None,
    "좌석 휠체어 이용가능" : "장애인 휠체어 이용가능",
    "출입구 휠체어 이용가능" : "장애인 휠체어 이용가능",
    "방문접수/출장" : None,
    "유아시설 (놀이방)" : None,
    "핸드드립" : None
}

mapping_very_good = {
    "직접 잘 구워줘요": None,
    "커피가 맛있어요": "음료가 맛있어요",
    "특별한 날 가기 좋아요": None,
    "아늑해요": None,
    "컨셉이 독특해요": None,
    "현지 맛에 가까워요": None,
    "혼술하기 좋아요": None,
    "야외공간이 멋져요": None,
    "오래 머무르기 좋아요": None,
    "반찬이 잘 나와요": None,
    "품질이 좋아요": None,
    "종류가 다양해요": None,
    "빵이 맛있어요": None,
    "시설이 깔끔해요": None,
    "게임 종류가 다양해요": None,
    "침구가 좋아요": None,
    "조용히 쉬기 좋아요": None,
    "깨끗해요": None,
    "메뉴 구성이 알차요": None,
    "차가 맛있어요": None,
    "마사지가 시원해요": None,
    "맞춤 케어를 잘해줘요": None,
    "분위기가 편안해요": None,
    "뷰가 좋아요": None
}




In [3]:
df = pd.read_csv("SQL_DB.csv",encoding="utf-8-sig")

In [4]:
df.head(3)

,id,size,road_address,name,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"[['""음식이 맛있어요""', 293], ['""인테리어가 멋져요""', 168], ['...","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식
1,2,610.52,"서울특별시 마포구 양화로 81, H 스퀘어 102호,103호,104호,105호 (서교동)",탭샵바 합정점,192565.8569,449977.7782,0507-1467-0765,화: 11:00 - 24:00; 수: 11:00 - 24:00; 목: 11:00 -...,131,"['와인 페어링', '전문 소믈리에', '포장', '단체 이용 가능', '배달', ...",무료 주차 가능,"[['""술이 다양해요""', 111], ['""음식이 맛있어요""', 99], ['""인테...","['룸', '단체석', '카운터석', '테라스', '좌식', '1인석', '연인석']","[[""루꼴라 치즈 떡볶이"", 9900]]",한식
2,3,48.37,"서울특별시 마포구 와우산로29마길 7-8, 지1층 우측호 (서교동)",연남골목냉면,193472.1057,450345.4305,NaN,NaN,38,[],NaN,"[['""음식이 맛있어요""', 58], ['""재료가 신선해요""', 15], ['""양이...",[],"[[""명태회냉면"", 10000], [""물냉면"", 8000], [""비빔냉면"", 800...",한식


In [5]:
import ast

# 'facilities'랑 'very_good' 컬럼이 문자열 리스트라면 변환
df['facilities'] = df['facilities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['very_good'] = df['very_good'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [10]:
df["facilities"][1][3]

'단체 이용 가능'

In [19]:
def clean_column_values(row, mapping):
    # 1) row가 리스트나 배열인지 먼저 확인
    if isinstance(row, list):
        # 리스트이면 pd.isna(row) 대신 바로 매핑만 수행
        return [mapping.get(item, item) for item in row if mapping.get(item, item) is not None]
    
    # 2) 만약 스칼라(단일 값)라면 pd.isna(row) 체크 가능
    if pd.isna(row):
        return []
    
    # 3) row가 문자열이면 ast.literal_eval로 리스트 변환 시도
    if isinstance(row, str):
        import ast
        try:
            row = ast.literal_eval(row)
        except (ValueError, SyntaxError):
            return []
        # 변환에 성공했다면 리스트가 되었을 것이므로 매핑 수행
        if isinstance(row, list):
            return [mapping.get(item, item) for item in row if mapping.get(item, item) is not None]
        else:
            # 변환했는데 리스트가 아니면 빈 리스트
            return []
    
    # 그 외 타입은 그냥 빈 리스트
    return []

# 시리즈의 각 원소에 대해 함수 적용
df['facilities'] = df['facilities'].apply(lambda x: clean_column_values(x, mapping_facilities))
df['very_good']  = df['very_good'].apply(lambda x: clean_column_values(x, mapping_very_good))

print(df)

      id    size                                       road_address  \
0      1   63.02                    서울특별시 마포구 양화로6길 57-12, 1층 (서교동)   
1      2  610.52  서울특별시 마포구 양화로 81, H 스퀘어 102호,103호,104호,105호 (서교동)   
2      3   48.37              서울특별시 마포구 와우산로29마길 7-8, 지1층 우측호 (서교동)   
3      4   54.00                서울특별시 마포구 독막로3길 24-10, 지1층 A호 (서교동)   
4      5   93.82                     서울특별시 마포구 와우산로27길 62, 2층 (서교동)   
..   ...     ...                                                ...   
583  584   28.95                    서울특별시 마포구 월드컵로5길 74 (합정동, 1층일부)   
584  585   59.46                   서울특별시 마포구 독막로 55 (합정동, 나동 1층 일부)   
585  586   75.11                     서울특별시 마포구 성지길 36-20, 지1층 (합정동)   
586  587  137.00  서울특별시 마포구 월드컵로1길 14, 지1층 B118-1, B118-2호 (합정동,...   
587  588   90.47                  서울특별시 마포구 양화진4길 17, 동원빌딩 4층 (합정동)   

            name     latitude    longitude           phone  \
0      라운지목화 합정관  192669.3832  449648.5161  0507-1469-7338   
1        탭샵바 합정점  19256

In [29]:
df['very_good'] = df['very_good'].apply(lambda x: [item[0] for item in x] if isinstance(x, list) else [])
df['seat_info'] = df['seat_info'].apply(lambda x: [item[0] for item in x] if isinstance(x, list) else [])

In [22]:
df.to_csv("SQL_DB_updated.csv",index=False,encoding="utf-8-sig")

In [23]:
df.head(1)

,id,size,road_address,name,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"[단체 이용 가능, 예약, 무선 인터넷]",주차 불가,"[""음식이 맛있어요"", ""인테리어가 멋져요"", ""친절해요"", ""대화하기 좋아요""]","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식


In [78]:
df = pd.read_csv("SQL_DB_updated.csv",encoding="utf-8-sig")

In [79]:
df.head(1)

,id,size,road_address,name,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"['""음식이 맛있어요""', '""인테리어가 멋져요""', '""친절해요""', '""대화하기...","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식


In [80]:
# 변환할 컬럼들
columns_to_fix = ["very_good"]

# 리스트 변환 + 이중 따옴표 제거
for col in columns_to_fix:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)
    df[col] = df[col].apply(lambda x: [i.strip('"') if isinstance(i, str) else i for i in x] if isinstance(x, list) else x)

# 결과 확인
df.head()

,id,size,road_address,name,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"[음식이 맛있어요, 인테리어가 멋져요, 친절해요, 대화하기 좋아요]","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식
1,2,610.52,"서울특별시 마포구 양화로 81, H 스퀘어 102호,103호,104호,105호 (서교동)",탭샵바 합정점,192565.8569,449977.7782,0507-1467-0765,화: 11:00 - 24:00; 수: 11:00 - 24:00; 목: 11:00 -...,131,"['와인 페어링', '와인 페어링', '포장', '단체 이용 가능', '배달', '...",무료 주차 가능,"[술이 다양해요, 음식이 맛있어요, 인테리어가 멋져요, 음악이 좋아요]","['룸', '단체석', '카운터석', '테라스', '좌식', '1인석', '연인석']","[[""루꼴라 치즈 떡볶이"", 9900]]",한식
2,3,48.37,"서울특별시 마포구 와우산로29마길 7-8, 지1층 우측호 (서교동)",연남골목냉면,193472.1057,450345.4305,NaN,NaN,38,[],NaN,"[음식이 맛있어요, 재료가 신선해요, 양이 많아요, 혼밥하기 좋아요]",[],"[[""명태회냉면"", 10000], [""물냉면"", 8000], [""비빔냉면"", 800...",한식
3,4,54.00,"서울특별시 마포구 독막로3길 24-10, 지1층 A호 (서교동)",유아하,192626.3225,449645.0116,0507-1432-7841,수: 17:00 - 24:00; 목: 17:00 - 24:00; 금: 17:00 -...,60,"['콜키지 가능', '생일 혜택', '단체 이용 가능']",주차 불가,"[음식이 맛있어요, 인테리어가 멋져요, 특별한 메뉴가 있어요, 친절해요]","['바테이블', '1인석', '테라스', '연인석', '카운터석', '좌식', '단...","[[""유아하 할머니의 훠궈"", 32900], [""우삼겹 후추탕"", 25000], [...",중식
4,5,93.82,"서울특별시 마포구 와우산로27길 62, 2층 (서교동)",증증일상,193340.6966,450339.4518,02-333-5888,화: 11:00 - 22:30; 수: 정기휴무 (매주 수요일); 목: 11:00 -...,476,"['포장', '배달', '무선 인터넷', '유아의자', '예약']",주차 불가,"[음식이 맛있어요, 친절해요, 특별한 메뉴가 있어요, 가성비가 좋아요]",[],"[[""꽌탕바오(육즙 샤오롱바오)"", 7300], [""성젠바오(육즙 군만두)"", 70...",중식


In [81]:
df.to_csv("SQL_DB_updated.csv",encoding="utf-8-sig",index=False)

In [64]:
df.head(1)

,id,size,road_address,name,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"['""음식이 맛있어요""', '""인테리어가 멋져요""', '""친절해요""', '""대화하기...","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식


In [67]:
# 고유한 값 확인할 컬럼들
columns_to_check = ["facilities", "parking", "very_good", "seat_info"]

# 각 컬럼의 고유한 값 출력
for col in columns_to_check:
    # 문자열이 리스트처럼 저장되어 있다면 변환
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)

    # NaN 제거 후 리스트 내부 요소까지 펼치기
    unique_values = df[col].explode().dropna().unique()

    print(f"Unique values in {col}:")
    print(unique_values)
    print("\n" + "-"*50 + "\n")

Unique values in facilities:
['단체 이용 가능' '예약' '무선 인터넷' '와인 페어링' '포장' '배달' '장애인 휠체어 이용가능' nan '콜키지 가능'
 '생일 혜택' '유아의자' '노키즈존' '비건 메뉴' '반려동물 동반' '대기공간' '테이크아웃 할인' '장애인 주차구역'
 '무한 리필' '글루텐프리 메뉴' '유기농 메뉴']

--------------------------------------------------

Unique values in parking:
['주차 불가' '무료 주차 가능' '유료 주차 가능' '주차 가능']

--------------------------------------------------

Unique values in very_good:
['"음식이 맛있어요"' '"인테리어가 멋져요"' '"친절해요"' '"대화하기 좋아요"' '"술이 다양해요"' '"음악이 좋아요"'
 '"재료가 신선해요"' '"양이 많아요"' '"혼밥하기 좋아요"' '"특별한 메뉴가 있어요"' '"가성비가 좋아요"'
 '"고기 질이 좋아요"' '"직접 잘 구워줘요"' '"기본 안주가 좋아요"' '"단체모임 하기 좋아요"' '"디저트가 맛있어요"'
 '"음료가 맛있어요"' '"커피가 맛있어요"' '"특별한 날 가기 좋아요"' '"매장이 넓어요"' '"매장이 청결해요"'
 '"아늑해요"' '"컨셉이 독특해요"' '"현지 맛에 가까워요"' '"혼술하기 좋아요"' '"야외공간이 멋져요"'
 '"오래 머무르기 좋아요"' '"반찬이 잘 나와요"' '"품질이 좋아요"' '"종류가 다양해요"' '"빵이 맛있어요"'
 '"시설이 깔끔해요"' '"게임 종류가 다양해요"' '"침구가 좋아요"' '"조용히 쉬기 좋아요"' '"깨끗해요"'
 '"메뉴 구성이 알차요"' '"차가 맛있어요"' nan '"마사지가 시원해요"' '"맞춤 케어를 잘해줘요"'
 '"분위기가 편안해요"' '"뷰가 좋아요"']

-------------------------

In [9]:
import pandas as pd

photo = pd.read_csv("photo_url.csv",encoding="utf-8-sig")
db = pd.read_csv("SQL_DB_updated.csv",encoding="utf-8-sig")

In [11]:
photo.head(1)
photo = photo[["id","photo_url","road_address","name"]]

In [13]:
db.head(1)
db = db.drop(columns=["road_address","name"])

In [14]:
merged_df = photo.merge(db, on = 'id')

In [15]:
merged_df.head()

,id,photo_url,road_address,name,size,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu,category
0,1,['https://search.pstatic.net/common/?autoRotat...,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,63.02,192669.3832,449648.5161,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"['음식이 맛있어요', '인테리어가 멋져요', '친절해요', '대화하기 좋아요']","['카운터석', '입식', '바테이블']","[[""아보카도크림새우"", 19900], [""깐풍기"", 18500], [""사천가지튀김...",중식
1,2,['https://search.pstatic.net/common/?autoRotat...,"서울특별시 마포구 양화로 81, H 스퀘어 102호,103호,104호,105호 (서교동)",탭샵바 합정점,610.52,192565.8569,449977.7782,0507-1467-0765,화: 11:00 - 24:00; 수: 11:00 - 24:00; 목: 11:00 -...,131,"['와인 페어링', '와인 페어링', '포장', '단체 이용 가능', '배달', '...",무료 주차 가능,"['술이 다양해요', '음식이 맛있어요', '인테리어가 멋져요', '음악이 좋아요']","['룸', '단체석', '카운터석', '테라스', '좌식', '1인석', '연인석']","[[""루꼴라 치즈 떡볶이"", 9900]]",한식
2,3,['https://search.pstatic.net/common/?autoRotat...,"서울특별시 마포구 와우산로29마길 7-8, 지1층 우측호 (서교동)",연남골목냉면,48.37,193472.1057,450345.4305,NaN,NaN,38,[],NaN,"['음식이 맛있어요', '재료가 신선해요', '양이 많아요', '혼밥하기 좋아요']",[],"[[""명태회냉면"", 10000], [""물냉면"", 8000], [""비빔냉면"", 800...",한식
3,4,['https://search.pstatic.net/common/?autoRotat...,"서울특별시 마포구 독막로3길 24-10, 지1층 A호 (서교동)",유아하,54.00,192626.3225,449645.0116,0507-1432-7841,수: 17:00 - 24:00; 목: 17:00 - 24:00; 금: 17:00 -...,60,"['콜키지 가능', '생일 혜택', '단체 이용 가능']",주차 불가,"['음식이 맛있어요', '인테리어가 멋져요', '특별한 메뉴가 있어요', '친절해요']","['바테이블', '1인석', '테라스', '연인석', '카운터석', '좌식', '단...","[[""유아하 할머니의 훠궈"", 32900], [""우삼겹 후추탕"", 25000], [...",중식
4,5,['https://search.pstatic.net/common/?autoRotat...,"서울특별시 마포구 와우산로27길 62, 2층 (서교동)",증증일상,93.82,193340.6966,450339.4518,02-333-5888,화: 11:00 - 22:30; 수: 정기휴무 (매주 수요일); 목: 11:00 -...,476,"['포장', '배달', '무선 인터넷', '유아의자', '예약']",주차 불가,"['음식이 맛있어요', '친절해요', '특별한 메뉴가 있어요', '가성비가 좋아요']",[],"[[""꽌탕바오(육즙 샤오롱바오)"", 7300], [""성젠바오(육즙 군만두)"", 70...",중식


In [16]:
merged_df.to_csv("final_sql.csv", encoding="utf-8-sig",index=False)